# SentryNet -- Modeling and Evaluation

In [ ]:
from sentrynet.config import DATA_DIR

TRANSACTION_PATH = DATA_DIR / "train_transaction.csv"
IDENTITY_PATH = DATA_DIR / "train_identity.csv"
DATA_AVAILABLE = TRANSACTION_PATH.exists()

if not DATA_AVAILABLE:
    print(f"Dataset not found at {TRANSACTION_PATH}. Download the IEEE-CIS "
          "Fraud Detection dataset from Kaggle and place it under data/ to run this notebook.")

## Temporal train/test split, merchant risk encoding (train-only fit), and training

In [ ]:
if DATA_AVAILABLE:
    from sentrynet.data.split import temporal_split
    from sentrynet.features.merchant_risk import MerchantRiskEncoder
    from sentrynet.modeling.train import train_model

    train_df, test_df = temporal_split(df, time_col="TransactionDT")

    risk_encoder = MerchantRiskEncoder(category_col="ProductCD").fit(train_df)
    train_df = train_df.assign(merchant_risk=risk_encoder.transform(train_df))
    test_df = test_df.assign(merchant_risk=risk_encoder.transform(test_df))

    feature_cols = ["TransactionAmt", "dist1", "dist2", "velocity_1h", "time_since_last",
                     "addr_changed", "merchant_risk",
                     "device_fingerprint_degree", "card_entity_id_degree"]
    model = train_model(train_df[feature_cols], train_df["isFraud"])

## Comparison: SMOTE vs. scale_pos_weight (evaluated for completeness, not used as primary)

In [ ]:
if DATA_AVAILABLE:
    from imblearn.over_sampling import SMOTE
    from sentrynet.modeling.evaluate import pr_auc
    import xgboost as xgb

    X_resampled, y_resampled = SMOTE(random_state=42).fit_resample(
        train_df[feature_cols], train_df["isFraud"]
    )
    smote_model = xgb.XGBClassifier(eval_metric="aucpr", random_state=42)
    smote_model.fit(X_resampled, y_resampled)
    smote_scores = smote_model.predict_proba(test_df[feature_cols])[:, 1]
    print("PR-AUC (SMOTE):", pr_auc(test_df["isFraud"], smote_scores))
    print("PR-AUC (scale_pos_weight):", pr_auc(test_df["isFraud"], model.predict_proba(test_df[feature_cols])[:, 1]))

## Evaluation: PR-AUC and cost-based threshold

In [ ]:
if DATA_AVAILABLE:
    from sentrynet.modeling.evaluate import pr_auc, select_threshold_by_cost

    test_scores = model.predict_proba(test_df[feature_cols])[:, 1]
    print("PR-AUC:", pr_auc(test_df["isFraud"], test_scores))
    best_threshold, best_cost = select_threshold_by_cost(
        test_df["isFraud"], test_scores, cost_fp=5.0, cost_fn=100.0
    )
    print("Best threshold:", best_threshold, "cost:", best_cost)